- prompt caching
    - https://sankalp.bearblog.dev/how-prompt-caching-works/

### chat template

- `--chat-template-content-format`: chat 模板里 content 这个字段到底长什么样
    - "string" 模式：传统纯文本，`{"role": "user", "content": "Hello world"}`
        - `{{ message.content }}`
    - "openai" 模式：OpenAI schema，多段内容列表（支持 text/image 等 multimodal）
        ```json
        {
          "role": "user",
          "content": [
            { "type": "text", "text": "Hello world!" },
            { "type": "image_url", "image_url": { "url": "..." } }
          ]
        }
        {% for part in message.content %}
          {% if part.type == "text" %}{{ part.text }}{% endif %}
        {% endfor %}
        ```
    - `auto`: default
        - vLLM 会尝试自动检测这个模型的 chat template 期望的格式，并在 log 里打印类似：
            - `Detected the chat template content format to be 'openai' 或 string。`

- huggingface chat template playground
    - https://huggingface.co/spaces/huggingfacejs/chat-template-playground
        - 左侧选择对应模型 chat template
        - 右侧上方：
            - hello world example
            - reasoning example
            - tool use example

### 确定性推理  

- https://docs.vllm.ai/en/latest/usage/reproducibility/
    - https://github.com/vllm-project/vllm/blob/main/examples/offline_inference/reproducibility.py
    - https://github.com/vllm-project/vllm/issues/29581#issuecomment-3608331306
- vllm Continuous Batching
    - GPU 的浮点运算不确定性（Non-determinism），浮点数加法不满足结合律：$(A+B)+C \neq A+(B+C)$
    - 请求可能独自处理，也可能和别人的请求拼在一起处理。
    - 当 Batch Size 变化时，CUDA Kernel（特别是 FlashAttention）的归约（Reduction）顺序会发生微小变化。这会导致输出的 Logits 在小数点后 4-5 位出现差异。
- global seed vs. generation seed
    - The seed parameter in vLLM is used to control the random states for various random number generators. If a specific seed value is provided, the random states for random, np.random, and torch.manual_seed will be set accordingly.
    - In V1, the seed parameter defaults to 0 which sets the random state for each worker, so the results will remain consistent for each vLLM run even if temperature > 0.
    - limits-of-rlvr
        - LLM(seed)
- MULTIPROCESSING（默认开启）
    - VLLM_ENABLE_V1_MULTIPROCESSING=0, 禁用多进程，实现调度的确定性；
- Batch Invariance
    - Outputs will be deterministic regardless of batch size
    - Batch invariance currently requires NVIDIA GPUs with compute capability 9.0 or higher:
        - H-series: H100, H200
        - B-series: B100, B200

### v1 engine

> vLLM API server version

### cuda graph

CUDA Graph 的引入主要是为了解决 CPU Launch Overhead（内核启动开销） 的瓶颈问题。在 LLM 推理的 Decode 阶段（逐个 token 生成），由于 batch size 通常较小且计算量较少，GPU 执行一个 Kernel 的时间可能比 CPU 下发这个 Kernel 的指令时间还要短。如果不使用 CUDA Graph，CPU 就会成为瓶颈，GPU 会出现大量空闲等待时间。

### reasoning model

- vLLM ≥ 0.10.0：不要再写 `--enable-reasoning`，只需要 `--reasoning-parser qwen3` 即开启 `reasoning`。
- vLLM 0.9.x：可以按 Qwen 官网示例同时加 `--enable-reasoning --reasoning-parser qwen3`。
- 如果不指定 `--reasoning-parser`
    - 无法切出 reasoning content 和一般的 content
    - 1203 对于 qwen3-30b-a3b-thinking, `--reasoning-parser deepseek_r1` 比较稳健
    - 如果指定了 `--reasoning-parser`，将会导致在完整地输出完 `<think></think>`之后，才会开始流式（stream）
- thinking budget

```
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "qwen3-14b",
    "messages": [
      {"role": "user", "content": "2+2 等于几？请展示推理过程。"}
    ],
    "max_tokens": 512
  }' | jq '.choices[0].message'
```

```
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "qwen3-14b",
    "messages": [
      {"role": "user", "content": "2+2 等于几？"}
    ],
    "max_tokens": 64,
    "chat_template_kwargs": {"enable_thinking": false}
  }' | jq '.choices[0].message'
```

- 软开关 (`/think`, `/no_think`)

```
{"role": "user", "content": "/think 详细证明 7543+3542 的计算过程"}
{"role": "user", "content": "/no_think 现在仅给出 7543+3542 的结果，不用过程"}
```

### structure output of reasoning model